# YouTube Profile Face Analysis

This notebook performs demographic analysis of users and channel creators using their YouTube profile pictures.

**Pipeline**: Download profile pictures → Face detection & alignment (DeepFace) → Age and gender analysis (InsightFace)

**Output**: CSV files with estimated age and gender for users and channels, for demographic analysis.

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os
import re
from concurrent.futures import ThreadPoolExecutor
import requests
from functools import partial
from deepface import DeepFace
import random
import cv2
import tensorflow as tf
from insightface.app import FaceAnalysis

# Project and data folders (make generic at top)
PROJECT_ROOT = os.path.abspath(os.getcwd())
CLEANED_DATA_DIR = os.path.join(PROJECT_ROOT, 'cleaned_data')
MODEL_PRED_DIR = os.path.join(PROJECT_ROOT, 'modelo', 'prediction')
PROFILE_FILES_DIR = os.path.join(PROJECT_ROOT, 'profile_files')
os.makedirs(PROFILE_FILES_DIR, exist_ok=True)

USERS_IMAGE_PATH = os.path.join(PROFILE_FILES_DIR, 'profile_pictures')
CHANNELS_IMAGE_PATH = os.path.join(PROFILE_FILES_DIR, 'channels_pictures')
os.makedirs(USERS_IMAGE_PATH, exist_ok=True)
os.makedirs(CHANNELS_IMAGE_PATH, exist_ok=True)

# logs for downloads
LOG_USERS = os.path.join(PROFILE_FILES_DIR, 'users_log.txt')
LOG_CHANNELS = os.path.join(PROFILE_FILES_DIR, 'channels_log.txt')

# folders for faces and logs
USERS_FACES_PATH = os.path.join(PROFILE_FILES_DIR, 'profile_final')
CHANNELS_FACES_PATH = os.path.join(PROFILE_FILES_DIR, 'channels_final')
os.makedirs(USERS_FACES_PATH, exist_ok=True)
os.makedirs(CHANNELS_FACES_PATH, exist_ok=True)

LOG_FACES_USERS = os.path.join(PROFILE_FILES_DIR, 'log_faces_users.txt')
LOG_FACES_CHANNELS = os.path.join(PROFILE_FILES_DIR, 'log_faces_channels.txt')

# output CSVs for attributes
CSV_USERS_PATH = os.path.join(PROFILE_FILES_DIR, 'user_attributes_unique.csv')
CSV_CHANNELS_PATH = os.path.join(PROFILE_FILES_DIR, 'channels_attributes_unique.csv')

---
## 1. Profile Picture Download

Downloads profile pictures for users (commenters) and channels (creators) from YouTube:
- **URL Resize**: Increase resolution to 400x400 for better detection
- **Parallelization**: ThreadPoolExecutor with 20-30 workers for concurrent downloads
- **Logging**: Avoid re-downloading already processed images
- **Data**: Extracts URLs from profile_picture_url and author_profile_image_url fields in datasets

In [ ]:
df_channels = pd.read_csv(os.path.join(CLEANED_DATA_DIR, 'filtered_channels.csv'))
df_comments = pd.read_csv(os.path.join(MODEL_PRED_DIR, 'final_comments_match_cleaned_pred.csv'))

In [ ]:
channels_image_url = df_channels[['channel_id', 'profile_picture_url']].drop_duplicates(subset='channel_id').rename(columns={'channel_id': 'id', 'profile_picture_url': 'url'})
users_image_url = df_comments[['author_channel_id', 'author_profile_image_url']].drop_duplicates(subset='author_channel_id').rename(columns={'author_channel_id': 'id', 'author_profile_image_url': 'url'})

In [ ]:
len(channels_image_url), len(users_image_url)

In [ ]:
def resize_url(url, new_size=400):
    if pd.isna(url):
        return url
    return re.sub(r'([=\-/]s)\d+', rf'\g<1>{new_size}', url)

channels_image_url['url'] = channels_image_url['url'].apply(resize_url)
users_image_url['url'] = users_image_url['url'].apply(resize_url)

In [ ]:
len(channels_image_url), len(users_image_url)

In [ ]:
def load_log(log_file):
    if not os.path.exists(log_file):
        return []
    with open(log_file, 'r') as log_out:
        ids = [l.strip() for l in log_out.read().split()]
    return ids

In [ ]:
def download_from_url(data, folder_path, log_file, urls_processed):
    img_id, url = data
    save_name = str(img_id)

    if not url or pd.isna(url):
        return

    if save_name in urls_processed:
        return

    os.makedirs(folder_path, exist_ok=True)
    save_path = os.path.join(folder_path, save_name + ".jpg")

    try:
        response = requests.get(url, timeout=(5, 10))

        if response.status_code == 200:
            with open(save_path, 'wb') as file:
                file.write(response.content)

            with open(log_file, 'a') as log:
                log.write(save_name + '\n')

    except Exception:
        pass

In [ ]:
download_data_users = list(zip(users_image_url['id'], users_image_url['url']))
download_data_channels = list(zip(channels_image_url['id'], channels_image_url['url']))

users_done = set(load_log(LOG_USERS))
channels_done = set(load_log(LOG_CHANNELS))

In [ ]:
func_users = partial(
    download_from_url,
    folder_path=USERS_IMAGE_PATH,
    log_file=LOG_USERS,
    urls_processed=users_done,
)

with ThreadPoolExecutor(max_workers=30) as executor:
    list(tqdm(executor.map(func_users, download_data_users), total=len(download_data_users)))

In [ ]:
func_channels = partial(
    download_from_url,
    folder_path=CHANNELS_IMAGE_PATH,
    log_file=LOG_CHANNELS,
    urls_processed=channels_done,
)

with ThreadPoolExecutor(max_workers=20) as executor:
    list(tqdm(executor.map(func_channels, download_data_channels), total=len(download_data_channels)))

---
## 2. Demographic Attribute Analysis

Extracts age and gender from faces using InsightFace (buffalo_l model):
- **Detection**: minimum det_score 0.80 to filter drawings/avatars
- **Age Filter**: Remove profiles with age < 15 (possible noise)
- **Validation**: Discard images without faces or with multiple faces
- **Extracted Attributes**:
  - **age**: estimated age
  - **gender**: M (male) or F (female)
- **Output**: CSVs with ID, age and gender for each processed user and channel

In [ ]:
# create CSV files if they do not exist
if not os.path.exists(CSV_USERS_PATH):
    with open(CSV_USERS_PATH, 'w') as f:
        f.write('id,age,gender\n')
    df_users_att = pd.read_csv(CSV_USERS_PATH)
else:
    df_users_att = pd.read_csv(CSV_USERS_PATH)


if not os.path.exists(CSV_CHANNELS_PATH):
    with open(CSV_CHANNELS_PATH, 'w') as f:
        f.write('id,age,gender\n')
    df_channels_att = pd.read_csv(CSV_CHANNELS_PATH)
else:
    df_channels_att = pd.read_csv(CSV_CHANNELS_PATH)

In [ ]:
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(224, 224))

def get_facial_attributes(path_img, path_output, set_ids):
    filename = os.path.basename(path_img)
    uid = os.path.splitext(filename)[0]

    if uid in set_ids:
        return
    
    img = cv2.imread(path_img) 
    if img is None:
        return

    faces = app.get(img)
    if len(faces) == 0 or len(faces) > 1:
        return  # discard images with no face or with multiple faces
    
    # when there are multiple faces, the largest can be chosen
    # f = max(faces, key=lambda face: (face.bbox[2] - face.bbox[0]) * (face.bbox[3] - face.bbox[1]))
    f = faces[0]
    
    # threshold to reduce illustrations/avatars
    if f.det_score < 0.80:
        return

    # consider very small ages as noise
    if f.age < 15:
        return

    # InsightFace returns gender as numeric (1/0)
    gender = "M" if f.gender == 1 else "F"
    age = f.age

    with open(path_output, 'a') as o:
        o.write(f"{uid},{age},{gender}\n")

In [ ]:
all_faces_users = [os.path.join(USERS_IMAGE_PATH, f) for f in os.listdir(USERS_IMAGE_PATH) if f.lower().endswith('.jpg')]
ids = set(df_users_att['id'].unique()) if not df_users_att.empty else set()

for img in tqdm(all_faces_users):
    get_facial_attributes(img, CSV_USERS_PATH, ids)

In [ ]:
all_faces_channels = [os.path.join(CHANNELS_IMAGE_PATH, f) for f in os.listdir(CHANNELS_IMAGE_PATH) if f.lower().endswith('.jpg')]
ids = set(df_channels_att['id'].unique()) if not df_channels_att.empty else set()

for img in tqdm(all_faces_channels):
    get_facial_attributes(img, CSV_CHANNELS_PATH, ids)